---
**Probability Distributions in Python**
Data Analysis Course · Week 5
---

This notebook is the Python equivalent of the R Markdown `_04_probability_distributions.Rmd`.
Topics: **random variables**, `scipy.stats`'s `pdf`/`cdf`/`ppf`/`rvs` function family (Python's
equivalent of R's `d`/`p`/`q`/`r` prefixes), and the Binomial, Poisson, and Normal distributions.

Work through it cell by cell — run each code cell with **Shift+Enter**.

**Required packages:** `numpy`, `pandas`, `matplotlib`, `scipy`
```
pip install numpy pandas matplotlib scipy
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

## 0 – Recap of the previous sheet

Last time we learned about unsupervised learning — hierarchical clustering and PCA. Today we move
to another milestone of statistics: **probability distributions**.

## 1 – Introduction and objectives

In the lectures you learned about major distribution types (Normal, Binomial, Negative Binomial,
Poisson, Student's t, ...). Here you will experiment with them practically, make calculations, plot
results, and explore relations between distributions.

## 2 – Let's get started

R has one function *family* per distribution, with a prefix: **p** (cumulative probability),
**d** (density), **q** (quantile), **r** (random draw) — e.g. `pnorm`, `dnorm`, `qnorm`, `rnorm`.

`scipy.stats` groups these into **one distribution object** per distribution, with methods instead
of prefixes:

| R prefix | scipy.stats method | Meaning |
|---|---|---|
| `p...()` | `.cdf()` | cumulative probability |
| `d...()` | `.pdf()` (continuous) / `.pmf()` (discrete) | density / probability mass |
| `q...()` | `.ppf()` | quantile (inverse CDF) |
| `r...()` | `.rvs()` | random draw |

e.g. R's `pnorm`/`dnorm`/`qnorm`/`rnorm` all become `scipy.stats.norm.cdf`/`.pdf`/`.ppf`/`.rvs`.

### Practice on the various functions

In [ ]:
# R: pnorm(1:4, mean=2, sd=1)  — cumulative probability ("p" like cumulative P-robability)
print(stats.norm.cdf([1, 2, 3, 4], loc=2, scale=1))
print(stats.norm.cdf([1, 2, 3, 4], loc=2, scale=2))
print(stats.norm.cdf([1, 2, 3, 4], loc=4, scale=1))

# Do you understand why the cumulative probabilities change the way they do?

In [ ]:
# R: qnorm(c(0.25,0.5,0.75), mean=2, sd=1)  — inverse CDF ("q" like quantile)
print(stats.norm.ppf([0.25, 0.5, 0.75], loc=2, scale=1))
print(stats.norm.ppf([0.25, 0.5, 0.75], loc=2, scale=2))
print(stats.norm.ppf([0.25, 0.5, 0.75], loc=4, scale=1))

# Try with 1.0 (=100%) on any distribution. Can you explain the result?

In [ ]:
# "d" like density probability. For continuous distributions the density at one exact point isn't
# a probability by itself — it's used directly for DISCRETE distributions like the binomial (pmf).

# R: dbinom(5, size=5, prob=0.5)  — probability of 5 out of 5
print(stats.binom.pmf(5, n=5, p=0.5))

# probability of NOT getting 5 out of 5
print(1 - stats.binom.pmf(5, n=5, p=0.5))
# or, using the CDF up to 4 successes:
print(stats.binom.cdf(4, n=5, p=0.5))

# What is the probability of getting 5 out of 10? And NOT getting 5 out of 10?

Suppose body height follows a normal distribution with mean = 1.75 m, sd = 0.15. What is the probability of being taller than 1.9 m? Smaller than 1.6 m?

In [ ]:
## taller than 1.9
print(1 - stats.norm.cdf(1.9, loc=1.75, scale=0.15))
# scipy has no "lower.tail" argument — just subtract from 1 for the upper tail

## smaller than 1.6
print(stats.norm.cdf(1.6, loc=1.75, scale=0.15))
# CDF is "lower tail" (P(X <= x)) by default, exactly like R's default lower.tail=TRUE

In [ ]:
x = np.arange(1, 2.51, 0.01)
y = stats.norm.pdf(x, loc=1.75, scale=0.15)
plt.plot(x, y, color="red", linewidth=2)
plt.axvline(1.6, linestyle=":", linewidth=2)
plt.axvline(1.9, linestyle=":", linewidth=2)
plt.show()

In [ ]:
# R: r-functions generate random DRAWS instead of returning probabilities/values
## normal distribution
x = stats.norm.rvs(size=1000, loc=10, scale=5, random_state=0)
plt.hist(x, bins=20)
plt.show()

# Can you generate a Poisson distribution and a binomial distribution?
# stats.poisson.rvs(mu=..., size=...) / stats.binom.rvs(n=..., p=..., size=...)

## 3 – Normal/Gaussian distribution

$$P(x) = \frac{1}{{\sigma \sqrt {2\pi } }} \cdot e ^ \frac{-(x- \mu)^2}{{2\sigma ^2 }}$$

where $\mu$ is the mean and $\sigma$ is the standard deviation. The **standard normal distribution**
is the special case $\mu = 0$, $\sigma = 1$.

### 3.1 – Visualization

Let's generate three normal distributions with different means and visualize them together.

In [ ]:
x = np.arange(-10, 30.1, 0.1)
d1 = stats.norm.pdf(x, loc=0, scale=1)
d2 = stats.norm.pdf(x, loc=10, scale=1)
d3 = stats.norm.pdf(x, loc=20, scale=1)

r1 = stats.norm.rvs(size=1000, loc=0, scale=1, random_state=0)
r2 = stats.norm.rvs(size=1000, loc=10, scale=1, random_state=1)
r3 = stats.norm.rvs(size=1000, loc=20, scale=1, random_state=2)

bins = np.arange(-10, 30.5, 0.5)
plt.hist(r1, bins=bins, density=True, alpha=0.5)
plt.hist(r2, bins=bins, density=True, alpha=0.5)
plt.hist(r3, bins=bins, density=True, alpha=0.5)

plt.plot(x, d1, color="red", linewidth=2)
plt.plot(x, d2, color="blue", linewidth=2)
plt.plot(x, d3, color="green", linewidth=2)
plt.ylim(0, 0.5)
plt.show()

# Play with mean/scale. What happens if you remove density=True from the histograms?

### 3.2 – Application on a real dataset

We use the Normal distribution to model TP53 gene expression in 586 lung cancer patients — TP53 is
the most commonly mutated gene across cancer types.

In [ ]:
tp53_exp = pd.read_csv(
    "https://www.dropbox.com/s/rwopdr8ycmdg8bd/TP53_expression_LungAdeno.txt?dl=1",
    sep="\t"
).iloc[:, 0:2]
tp53_exp.describe(include="all")

#### 3.2.1 – Data cleaning and central values

In [ ]:
# R: tp53.exp[!is.na(tp53.exp$TP53_expression),]
tp53_exp = tp53_exp[tp53_exp["TP53_expression"].notna()]
m_tp53 = tp53_exp["TP53_expression"].mean()
s_tp53 = tp53_exp["TP53_expression"].std()
print(m_tp53, s_tp53)

#### 3.2.2 – Modeling the data using a normal distribution

In [ ]:
# distribution of the measured data
tp53_exp["TP53_expression"].plot(kind="density", linewidth=3, xlim=(-1500, 6000))
plt.show()

In [ ]:
# Model prediction with the sample mean/sd
x = np.arange(0, 5000, 5)
d_pred = stats.norm.pdf(x, loc=m_tp53, scale=s_tp53)

tp53_exp["TP53_expression"].plot(kind="density", linewidth=3, xlim=(-1500, 6000))
plt.plot(x, d_pred, color="red", linewidth=3)
plt.show()

#### 3.2.3 – Data prediction using the model

Using a Normal distribution with the sample $\mu$ and $\sigma$:

- **(Q1)** Probability that TP53 expression is **less** than 1000?
- **(Q2)** Probability that TP53 expression is **greater** than 1000?

In [ ]:
# Q1
print(stats.norm.cdf(1000, loc=m_tp53, scale=s_tp53))
# Q2
print(1 - stats.norm.cdf(1000, loc=m_tp53, scale=s_tp53))

#### 3.2.4 – Evaluating the quality of the predictions

In [ ]:
# Real data equivalents
print((tp53_exp["TP53_expression"] < 1000).sum() / len(tp53_exp))
print((tp53_exp["TP53_expression"] > 1000).sum() / len(tp53_exp))

# Re-run with q = 100, 500, 4000, 4500. At which values does the model perform worse?
# HINT: look at the tails of the distribution!

In [ ]:
q_values = [100, 500, 1000, 4000, 4500]

model_rows = []
for q in q_values:
    predicted = stats.norm.cdf(q, loc=m_tp53, scale=s_tp53)
    measured = (tp53_exp["TP53_expression"] < q).sum() / len(tp53_exp)
    model_rows.append([predicted, measured])

model_df = pd.DataFrame(
    model_rows, index=[f"q={q}" for q in q_values], columns=["Predicted", "Measured"]
).T
model_df

In [ ]:
# Q1: TP53 expression at the 10th percentile?
print(stats.norm.ppf(0.1, loc=m_tp53, scale=s_tp53))
# Q2: TP53 expression at the 90th percentile?
print(stats.norm.ppf(0.9, loc=m_tp53, scale=s_tp53))

In [ ]:
tp53_exp["TP53_expression"].quantile([0.1, 0.9])   # compare to the model's predictions above

#### 3.2.5 – Graphical visualization

In [ ]:
x = np.arange(0, 5000, 5)
d_pred = stats.norm.pdf(x, loc=m_tp53, scale=s_tp53)

# Model and measured data, plus predicted vs. measured 0.1/0.9 quantiles
tp53_exp["TP53_expression"].plot(kind="density", linewidth=3, xlim=(-1500, 6000))
plt.plot(x, d_pred, color="red", linewidth=3)
for v in tp53_exp["TP53_expression"].quantile([0.1, 0.9]):
    plt.axvline(v, color="black")
for v in [stats.norm.ppf(0.1, loc=m_tp53, scale=s_tp53), stats.norm.ppf(0.9, loc=m_tp53, scale=s_tp53)]:
    plt.axvline(v, color="red", linestyle=":", linewidth=2)
plt.show()

# Appreciate the quality of the predictions! Compare the black and red vertical lines.
# Re-run with p = 0.25, 0.5, 0.75 etc.

#### 3.2.6 – Graphical visualization using a Q-Q plot

In [ ]:
q = np.arange(0, 1.01, 0.01)

q_observed = tp53_exp["TP53_expression"].quantile(q)
q_theoretical = stats.norm.ppf(q, loc=m_tp53, scale=s_tp53)

plt.scatter(q_theoretical, q_observed, s=10)
lims = [min(q_theoretical.min(), q_observed.min()), max(q_theoretical.max(), q_observed.max())]
plt.plot(lims, lims, color="red", linewidth=2)
plt.xlabel("Theoretical quantiles")
plt.ylabel("Observed quantiles")
plt.show()

---
## Exercises

### Exercise I

1. What is the TP53 expression observed at the 10th percentile? At the 90th percentile?
2. Other distributions become similar to the normal distribution under certain conditions. Plot a
   histogram of 1000 random draws from a Poisson distribution with `mu = 1, 10, 100, 1000`. What do
   you observe? *(Hint: `stats.poisson.rvs(mu=..., size=1000)`)*

In [ ]:
# Your code here:

### Exercise II

1. A patient has convulsions and needs an injection each time, averaging 3 episodes per day. How
   many syringes should the hospital prepare daily to cover 99.9% of scenarios?
   *(Hint: `stats.poisson.ppf(0.999, mu=3)`)*
2. What are the odds of 2 episodes or fewer? Of 2 episodes or more?

In [ ]:
# Your code here:

## 4 – Getting further (optional): Poisson distribution

$$P(x) = \frac{e^{-\lambda}\lambda^x}{x!}$$

where $\lambda$ is the mean (which also equals the variance) and $x$ is the number of events.

An experiment follows a Poisson distribution when outcomes are binary, the mean rate $\lambda$ in a
given interval is known and constant, and the probability of an event in a very small interval
approaches zero.

**Example:** gene mutation. Human mutation rate ≈ $10^{-8}$ mutations/bp/generation, genome size ≈
$3\times10^9$ bp → expected mutations per generation ≈ 30.

In [ ]:
# R: ppois(q = 10, lambda = 30)  — probability of observing 10 or fewer mutations
print(stats.poisson.cdf(10, mu=30))

In [ ]:
q = np.arange(0, 61)
plt.plot(q, stats.poisson.cdf(q, mu=30), marker="o", markersize=3)
plt.axvline(30)
plt.xlabel("Average Mutations")
plt.ylabel("Probability")
plt.show()

In [ ]:
p_pred = stats.poisson.pmf(np.arange(0, 61), mu=30)
plt.bar(np.arange(0, 61), p_pred)
plt.show()

# It's clearly much more likely to observe around 30 mutations than the extremes.

Can you make the same prediction with a normal distribution instead? *Hint:* $\lambda = mean = variance$, so $\sigma = \sqrt{\lambda}$. What difference do you see?

In [ ]:
q = np.arange(0, 61)
pd_ = stats.poisson.cdf(q, mu=30)
nd = stats.norm.cdf(q, loc=30, scale=np.sqrt(30))   # var = sd^2

plt.plot(q, pd_, color="red")
plt.plot(q, nd, color="grey")
plt.xlabel("Average Mutations")
plt.ylabel("Probability")
plt.show()

## Summary: What have we learned?

| R | scipy.stats equivalent | Purpose |
|---|--------|---------|
| `pnorm(x, mean, sd)` | `stats.norm.cdf(x, loc, scale)` | Cumulative probability |
| `dnorm(x, mean, sd)` | `stats.norm.pdf(x, loc, scale)` | Density |
| `qnorm(p, mean, sd)` | `stats.norm.ppf(p, loc, scale)` | Quantile (inverse CDF) |
| `rnorm(n, mean, sd)` | `stats.norm.rvs(size=n, loc, scale)` | Random draws |
| `dbinom(x, size, prob)` | `stats.binom.pmf(x, n, p)` | Binomial probability mass |
| `ppois(q, lambda)` | `stats.poisson.cdf(q, mu)` | Poisson cumulative probability |
| `dpois(x, lambda)` | `stats.poisson.pmf(x, mu)` | Poisson probability mass |
| `lower.tail = FALSE` | `1 - cdf(...)` | Upper-tail probability |
| `quantile(x, probs)` | `x.quantile([...])` | Sample quantiles |